# TC_05: FCNN with class imbalance perturbation
Change class imbalance at 10%, 30%, 50% levels. Run full diagnostic pipeline at each level and compare against TC_03 (Golden Baseline)

In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

sys.path.append('..')
os.chdir('..')
plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

from src.utils.config_loader import load_config
from src.utils.data_loader import load_tabular_data
from src.data_diagnostics.quality_checks import check_tabular_quality
from src.utils.visualization import plot_class_distribution, plot_correlation, plot_feature_boxplots
from src.utils.performance_tracker import PerformanceTracker
from src.models.tabular_models import train_fcnn, evaluate_model_fcnn
from src.inference_diagnostics.uncertainty import deep_ensemble_prediction_tab, mc_dropout_prediction_tab
from src.inference_diagnostics.explainability import shap_tab, lime_tab,calculate_consistency_tabular
from src.inference_diagnostics.flagging import assign_flags,evaluate_flags
from src.utils.visualization import plot_uncertainty_distribution, plot_flag_distribution
from src.utils.report_generator import generate_report,save_report
from src.data_diagnostics.perturbations import inject_class_imbalance_tab
from src.utils.config_loader import load_config, get_tabular_config, load_threshold
from src.utils.per_sample_logger import save_per_sample, build_experiment_id

config = load_config()
tracker = PerformanceTracker()

### Load clean data


In [2]:
X_train, X_test, y_train, y_test = load_tabular_data(config)
quality_report = check_tabular_quality(X_train, y_train, config)

# Identity for this notebook
dataset_short = get_tabular_config(config)['short_name']
model_name = 'fcnn'
test_case = 'TC05'


# Load seed
seed = config['random_seeds']['primary_seed']
ratios =  config['stage1_data_quality']['perturbations']['class_imbalance']['ratios']

# Golden baseline thresholds written by TC_03
mc_threshold = load_threshold(config, dataset_short, model_name, 'mc')
de_threshold = load_threshold(config, dataset_short, model_name, 'de')
print(f"Loaded thresholds - MC: {mc_threshold}, DE: {de_threshold}")

Loading dataset.
Dataset: UCI Adult Income
Duplicate rows kept (6374 found).
Found 6465 missing values. Filling with mode.
Data loaded for 39073 train, 9769 test
 Features: 12
EDA started for tabular data.
Samples: 39073, Features: 12
Class distribution: {0: 29724, 1: 9349}
Imbalance ratio: 0.3145
Duplicate rows: 5422
Total outlier count: 5915
EDA completed for UCI Adult Income
Loaded thresholds - MC: 0.0874, DE: 0.0364


### Run full pipeline for each perturbation levels

In [3]:
all_results = {}
for ratio in ratios:
    print(f"Class imbalance for ratio: {ratio * 100:.0f}%")

    tracker = PerformanceTracker()

    # Class imbalance
    X_train_corrupted, y_train_corrupted = inject_class_imbalance_tab(X_train, y_train, ratio, seed)
    corrupted_report = check_tabular_quality(X_train_corrupted, y_train_corrupted, config)
    plot_class_distribution(corrupted_report, f"FCNN {get_tabular_config(config)['name']} fault values fraction: {ratio * 100:.0f}% ", config)
    plot_correlation(corrupted_report, f"FCNN {get_tabular_config(config)['name']} fault values fraction: {ratio * 100:.0f}%", config)
    plot_feature_boxplots(X_train_corrupted, f"FCNN {get_tabular_config(config)['name']} fault values fraction: {ratio * 100:.0f}%", config)

    # Train FCNN with class imbalance
    tracker.start_performance_track()
    fcnn_model = train_fcnn(X_train_corrupted, y_train_corrupted, config)
    tracker.stop_performance_track(f"FCNN training class imbalance ratio: {ratio * 100:.0f}%")

    tracker.start_performance_track()
    fcnn_accuracy, fcnn_prediction, fcnn_report = evaluate_model_fcnn(fcnn_model, X_test, y_test, config)
    tracker.stop_performance_track(f"FCNN evaluation class imbalance ratio: {ratio * 100:.0f}%")

    # MC Dropout
    tracker.start_performance_track()
    mc_means_probabilities, mc_uncertainty = mc_dropout_prediction_tab(fcnn_model, X_test, config)
    tracker.stop_performance_track(f"FCNN MC Dropout class imbalance ratio: {ratio * 100:.0f}%")

    print(f"MC Dropout uncertainty status:")
    print(f"Mean: {mc_uncertainty.mean()}")
    print(f"Max: {mc_uncertainty.max()}")
    print(f" 90th percentile (threshold): {mc_threshold}")

    plot_uncertainty_distribution(mc_uncertainty, f"FCNN MC Dropout class imbalance ratio: {ratio * 100:.0f}% ", mc_threshold, config)

    # Deep Ensemble
    tracker.start_performance_track()
    de_means_probabilities, de_uncertainty = deep_ensemble_prediction_tab(train_fcnn, X_train_corrupted, y_train_corrupted, X_test, config)
    tracker.stop_performance_track(f"FCNN Deep Ensemble class imbalance ratio: {ratio * 100:.0f}%")

    print(f"Deep Ensembl uncertainty status:")
    print(f"Mean: {de_uncertainty.mean()}")
    print(f"Max: {de_uncertainty.max()}")
    print(f" 90th percentile (threshold): {de_threshold}")

    plot_uncertainty_distribution(de_uncertainty, f"FCNN Deep Ensemble class imbalance ratio: {ratio * 100:.0f}%", de_threshold, config)

    # SHAP
    tracker.start_performance_track()
    shap_values, shap_samples = shap_tab(fcnn_model, X_train_corrupted, X_test, config, is_pytorch = True)
    tracker.stop_performance_track(f"FCNN SHAP class imbalance ratio: {ratio * 100:.0f}%")
    print(f"SHAP values shape: {shap_values.shape}")

    # LIME
    tracker.start_performance_track()
    lime_explanations, lime_samples = lime_tab(fcnn_model, X_train_corrupted, X_test, config, is_pytorch = True)
    tracker.stop_performance_track(f"FCNN LIME class imbalance ratio: {ratio * 100:.0f}%")
    print(f"LIME explanations: {len(lime_explanations)}")

    # Calculate consistency
    feature_names = list(X_train.columns)
    consistency_scores = calculate_consistency_tabular(shap_values, lime_explanations, feature_names, top=5)

    # MC Dropout with XAI flags
    mc_y_predictions = mc_means_probabilities[:len(consistency_scores)].argmax(axis = 1)
    mc_flags = assign_flags(mc_uncertainty[:len(consistency_scores)], consistency_scores, mc_threshold, 0.5)
    mc_flag_results = evaluate_flags(mc_flags, mc_y_predictions, y_test[:len(consistency_scores)])
    plot_flag_distribution(mc_flags, f"FCNN MC Dropout + XAI Flagging class imbalance ratio: {ratio * 100:.0f}%", config)

    # Deep Ensemble flags
    de_y_predictions = de_means_probabilities[:len(consistency_scores)].argmax(axis = 1)
    de_flags = assign_flags(de_uncertainty[:len(consistency_scores)], consistency_scores, de_threshold, 0.5)
    de_flag_results = evaluate_flags(de_flags, de_y_predictions, y_test[:len(consistency_scores)])
    plot_flag_distribution(de_flags, f"FCNN Deep Ensembles + XAI Flagging class imbalance ratio: {ratio * 100:.0f}%", config)

    # MC Dropout without XAI (constant consistency)
    con_consistency = np.ones(len(consistency_scores))
    mc_flags_uq_only = assign_flags(mc_uncertainty[:len(consistency_scores)], con_consistency, mc_threshold, 0.5)

    print(f"MC Dropout without XAI Flagging:")
    mc_only_flag_results = evaluate_flags(mc_flags_uq_only, mc_y_predictions, y_test[:len(consistency_scores)])
    plot_flag_distribution(mc_flags_uq_only, f"FCNN MC Dropout only class imbalance ratio: {ratio * 100:.0f}%", config)

    print(f"MC Dropout with XAI green accuracy: {mc_flag_results['GREEN']['accuracy']}")
    print(f"MC Dropout without XAI green accuracy: {mc_only_flag_results['GREEN']['accuracy']}")
    print(f"Improvement with XAI: {mc_flag_results['GREEN']['accuracy'] - mc_only_flag_results['GREEN']['accuracy']}")

    # Deep Ensemble without XAI (constant consistency)
    de_flags_uq_only = assign_flags(de_uncertainty[:len(consistency_scores)], con_consistency, de_threshold, 0.5)

    print(f"Deep Ensemble without XAI Flagging:")
    de_only_flag_results = evaluate_flags(de_flags_uq_only, de_y_predictions, y_test[:len(consistency_scores)])
    plot_flag_distribution(de_flags_uq_only, f"FCNN Deep Ensembles only class imbalance ratio: {ratio * 100:.0f}%", config)

    print(f"Deep Ensembles with XAI green accuracy: {de_flag_results['GREEN']['accuracy']}")
    print(f"Deep Ensembles without XAI green accuracy: {de_only_flag_results['GREEN']['accuracy']}")
    print(f"Improvement with XAI: {de_flag_results['GREEN']['accuracy'] - de_only_flag_results['GREEN']['accuracy']}")

    level_result = {
    'perturbation': 'class imbalance',
    'ratio': ratio,
    'model': 'FCNN',
    'accuracy': round(fcnn_accuracy, 4),
    'classification_report': fcnn_report,
    'clean_class_distribution': quality_report['class_distribution'],
    'corrupted_class_distribution': corrupted_report['class_distribution'],
    'quality_report': quality_report['feature_stats'],
    'corrupted_quality_report': corrupted_report['feature_stats'],
    'mc_uncertainty': {
        'mean': round(float(mc_uncertainty.mean()), 8),
        'max': round(float(mc_uncertainty.max()), 8),
        'threshold': mc_threshold
    },
    'de_uncertainty': {
        'mean': round(float(de_uncertainty.mean()), 8),
        'max': round(float(de_uncertainty.max()), 8),
        'threshold': de_threshold
    },
    'consistency':{
        'mean': round(float(consistency_scores.mean()), 4),
        'min': round(float(consistency_scores.min()), 4),
        'max': round(float(consistency_scores.max()), 4)
    },
    'flagging_mc': mc_flag_results,
    'flagging_de': de_flag_results,
    'flagging_mc_only': mc_only_flag_results,
    'mc_vs_mc_plus_xai': round(mc_flag_results['GREEN']['accuracy'] - mc_only_flag_results['GREEN']['accuracy'], 4),
    'flagging_de_only': de_only_flag_results,
    'de_vs_de_plus_xai': round(de_flag_results['GREEN']['accuracy'] - de_only_flag_results['GREEN']['accuracy'], 4),
    'performance': tracker.get_results()
    }

    all_results[f"{ratio * 100:.0f}"] = level_result

    # Per sample results for this imbalance level
    experiment_id = build_experiment_id(dataset_short, model_name, test_case, f"ci{int(ratio*100)}")
    save_per_sample(
        config,
        experiment_id,
        y_true=y_test,
        y_pred=mc_y_predictions,
        mc_uncertainty=mc_uncertainty,
        de_uncertainty=de_uncertainty,
        consistency=consistency_scores,
    )

    # Save individual report
    report_output = generate_report(config, f"{get_tabular_config(config)['name']} - FCNN class imbalance {ratio*100:.0f}%", stage1_result=level_result)
    save_report(report_output, f'{dataset_short.upper()}_TC_05_FCNN_Class_Imbalance_{int(ratio*100)}pct.json', config)


Class imbalance for ratio: 50%
Inject class imbalance, keeping 50% of minority class
Original minority count: 9349
New minority count: 4674
Majority count: 29724
EDA started for tabular data.
Samples: 34398, Features: 12
Class distribution: {0: 29724, 1: 4674}
Imbalance ratio: 0.1572
Duplicate rows: 4513
Total outlier count: 4857
EDA completed for UCI Adult Income
Saved: results/figures\class_distribution_fcnn_uci_adult_income_fault_values_fraction:_50%_.png
Saved: results/figures\correlation_fcnn_uci_adult_income_fault_values_fraction:_50%.png
Saved: results/figures\boxplot_fcnn_uci_adult_income_fault_values_fraction:_50%.png
FCNN training started.
Epoch 10/50
Epoch 20/50
Epoch 30/50
Epoch 40/50
 Early stopping at epoch 45
FCNN training completed.
FCNN training class imbalance ratio: 50%: Time:20.10s, Memory:551.89MB
Model evaluation started for FCNN.
{'0': {'precision': 0.8429982568274259, 'recall': 0.9761808639483246, 'f1-score': 0.9047143926166126, 'support': 7431.0}, '1': {'precis

  0%|          | 0/200 [00:00<?, ?it/s]

SHAP finished. Explained: 200 samples.
FCNN SHAP class imbalance ratio: 50%: Time:6.18s, Memory:13.92MB
SHAP values shape: (200, 12, 2)
LIME started.
Explained 50 / 200 samples.
Explained 100 / 200 samples.
Explained 150 / 200 samples.
Explained 200 / 200 samples.
LIME finished. Explained: 200 samples.
FCNN LIME class imbalance ratio: 50%: Time:1.84s, Memory:0.69MB
LIME explanations: 200
Calculate consistency tabular.
RED: Count: 2 Accuracy:100.0%
YELLOW: Count: 39 Accuracy:79.5%
GREEN: Count: 159 Accuracy:89.3%
Flagging system is not working, needs threshold adjustment
Saved: results/figures\flagging_distribution_fcnn_mc_dropout_+_xai_flagging_class_imbalance_ratio:_50%.png
RED: Count: 5 Accuracy:100.0%
YELLOW: Count: 38 Accuracy:68.4%
GREEN: Count: 157 Accuracy:91.7%
Flagging system is not working, needs threshold adjustment
Saved: results/figures\flagging_distribution_fcnn_deep_ensembles_+_xai_flagging_class_imbalance_ratio:_50%.png
MC Dropout without XAI Flagging:
RED: Count: 0 Acc

  0%|          | 0/200 [00:00<?, ?it/s]

SHAP finished. Explained: 200 samples.
FCNN SHAP class imbalance ratio: 30%: Time:6.37s, Memory:4.36MB
SHAP values shape: (200, 12, 2)
LIME started.
Explained 50 / 200 samples.
Explained 100 / 200 samples.
Explained 150 / 200 samples.
Explained 200 / 200 samples.
LIME finished. Explained: 200 samples.
FCNN LIME class imbalance ratio: 30%: Time:1.70s, Memory:0.00MB
LIME explanations: 200
Calculate consistency tabular.
RED: Count: 1 Accuracy:100.0%
YELLOW: Count: 17 Accuracy:76.5%
GREEN: Count: 182 Accuracy:87.9%
Flagging system is not working, needs threshold adjustment
Saved: results/figures\flagging_distribution_fcnn_mc_dropout_+_xai_flagging_class_imbalance_ratio:_30%.png
RED: Count: 1 Accuracy:100.0%
YELLOW: Count: 21 Accuracy:61.9%
GREEN: Count: 178 Accuracy:89.3%
Flagging system is not working, needs threshold adjustment
Saved: results/figures\flagging_distribution_fcnn_deep_ensembles_+_xai_flagging_class_imbalance_ratio:_30%.png
MC Dropout without XAI Flagging:
RED: Count: 0 Accu

  0%|          | 0/200 [00:00<?, ?it/s]

SHAP finished. Explained: 200 samples.
FCNN SHAP class imbalance ratio: 10%: Time:6.18s, Memory:4.02MB
SHAP values shape: (200, 12, 2)
LIME started.
Explained 50 / 200 samples.
Explained 100 / 200 samples.
Explained 150 / 200 samples.
Explained 200 / 200 samples.
LIME finished. Explained: 200 samples.
FCNN LIME class imbalance ratio: 10%: Time:1.87s, Memory:0.00MB
LIME explanations: 200
Calculate consistency tabular.
RED: Count: 3 Accuracy:100.0%
YELLOW: Count: 61 Accuracy:82.0%
GREEN: Count: 136 Accuracy:86.0%
Flagging system is not working, needs threshold adjustment
Saved: results/figures\flagging_distribution_fcnn_mc_dropout_+_xai_flagging_class_imbalance_ratio:_10%.png
RED: Count: 3 Accuracy:100.0%
YELLOW: Count: 66 Accuracy:77.3%
GREEN: Count: 131 Accuracy:88.5%
Flagging system is not working, needs threshold adjustment
Saved: results/figures\flagging_distribution_fcnn_deep_ensembles_+_xai_flagging_class_imbalance_ratio:_10%.png
MC Dropout without XAI Flagging:
RED: Count: 0 Accu